In [ ]:
# Uncomment if packages are not installed
# !pip install lets-plot json pandas numpy

import json
import pandas as pd
import numpy as np
from lets_plot import *
from lets_plot.mapping import as_discrete

LetsPlot.setup_html()


In [ ]:
DATA_PATH      = 'P2_S12_EMG1kHz.txt';
CH_META_PATH   = 'metadata.json';
TRIAL_META_PATH = 'P2_S12_meta.json';

In [ ]:
# load metadara
# Channel names 
with open(CH_META_PATH) as f:
    ch_meta = json.load(f)

# Build mapping: EMG1k_N -> channel name
ch_map = {f"EMG1k_{d['channelNumber']}": d["channelName"] for d in ch_meta}
print("Channel mapping:")
for k, v in ch_map.items():
    print(f"  {k}  →  {v}")

# Trial metadata 
with open(TRIAL_META_PATH) as f:
    trial_meta_raw = json.load(f)

trial_meta = pd.DataFrame(trial_meta_raw)[["trialID", "movementNumber", "trialNumber", "onsetTime"]]
print(f"\nTrial metadata: {len(trial_meta)} trials, movements: {sorted(trial_meta['movementNumber'].unique())}")
trial_meta.head(10)

In [ ]:
# loading emg data
FS = 1000  # sampling rate in Hz

df = pd.read_csv(DATA_PATH)

EMG_COLS = [c for c in df.columns if c.startswith("EMG")]

# Rename columns to anatomical names
df = df.rename(columns=ch_map)
CH_NAMES = [ch_map[c] for c in EMG_COLS]  # ordered list of channel names

# Add within-trial time axis
df["time_s"] = df.groupby("trialID").cumcount() / FS

# Merge in trial metadata (movementNumber, onsetTime)
df = df.merge(trial_meta[["trialID", "movementNumber", "onsetTime"]], on="trialID", how="left")

print(f"Loaded {len(df):,} rows | {df['trialID'].nunique()} trials | {len(CH_NAMES)} channels")
df.head()

In [ ]:
TRIAL_ID        = 7      # single trial to display (ignored when PLOT_MEAN=True)
MOVEMENT_NUMBER = 1      # movement to average over (used when PLOT_MEAN=True)
PLOT_MEAN       = False  # True → trial-mean for MOVEMENT_NUMBER
SHOW_ONSET      = True   # draw vertical line at movement onset

if PLOT_MEAN:
    subset = df[df["movementNumber"] == MOVEMENT_NUMBER]
    onset_time = subset["onsetTime"].mean()
    agg = subset.groupby("time_s")[CH_NAMES].mean().reset_index()
    data_to_plot = agg.melt(id_vars="time_s", var_name="channel", value_name="value")
    # Preserve channel order
    data_to_plot["channel"] = pd.Categorical(data_to_plot["channel"], categories=CH_NAMES, ordered=True)
    n_trials = subset["trialID"].nunique()
    title = f"Mean EMG — Movement {MOVEMENT_NUMBER}  (n={n_trials} trials)"
else:
    row = trial_meta[trial_meta["trialID"] == TRIAL_ID].iloc[0]
    onset_time = row["onsetTime"]
    mov_num    = int(row["movementNumber"])
    trial_num  = int(row["trialNumber"])
    trial_df   = df[df["trialID"] == TRIAL_ID].copy()
    data_to_plot = trial_df[["time_s"] + CH_NAMES].melt(
        id_vars="time_s", var_name="channel", value_name="value"
    )
    data_to_plot["channel"] = pd.Categorical(data_to_plot["channel"], categories=CH_NAMES, ordered=True)
    title = f"EMG — Trial {TRIAL_ID}  (Movement {mov_num}, Rep {trial_num})"

print(f"Title   : {title}")
print(f"Onset   : {onset_time:.3f} s")
print(f"Rows    : {len(data_to_plot):,}")

In [ ]:
#plotting all channels 

p = (
    ggplot(data_to_plot, aes(x="time_s", y="value", color="channel"))
    + geom_line(size=0.4, alpha=0.85, show_legend=False)
    + facet_wrap("channel", ncol=1, scales="free")  # numchans×1 grid, independent y-axes
    + scale_x_continuous(breaks=list(range(0, 9, 2)))
    + labs(title=title, x="Time (s)", y="Amplitude (µV)")
    + theme_bw()
    + theme(
        plot_title=element_text(size=13, face="bold", hjust=0.5),
        strip_background=element_rect(fill="#dce6f0"),
        strip_text=element_text(size=10, face="bold"),
        panel_grid_minor=element_blank(),
        panel_grid_major=element_line(color="#e0e0e0", size=0.4),
        axis_text=element_text(size=8),
    )
    + ggsize(900, 1200)
)

# Optionally overlay a vertical onset line
if SHOW_ONSET:
    p = p + geom_vline(xintercept=onset_time, linetype="dashed", color="#e63946", size=0.7)

p


In [ ]:
# saving
# ggsave(p, 'emg_channels.png', dpi=150)
# print('Saved → emg_channels.png')
